# Support Ticket Triage with Typed Multi-Agent Handoffs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibm-granite-community/mellea-cookbook/blob/main/recipes/SupportTicketTriage/SupportTicketTriage.ipynb)

This recipe builds a three-step support pipeline. Each step is a separate agent, and each handoff passes a typed object:

1. **Classify.** A generative step returns a `TicketTriage` with a category, priority, and reason.
2. **Research.** A tool-using agent looks up the customer's order in a database, using the `lookup_order` and `lookup_customer` tools.
3. **Reply.** A drafting step writes a customer reply. Requirements check that the reply quotes the real order status and doesn't promise anything the data doesn't support.

The structure is what keeps the pipeline reliable. A bad category, a missing order ID, or an invented order status fails a typed check, and Mellea repairs the step instead of passing the error downstream.

## What You'll Build

- A three-step support pipeline where each step is an agent with a typed output.
- A `react` research agent that calls `lookup_order` and `lookup_customer` tools against a SQLite database.
- A reply draft checked by requirements before it's returned.

## Prerequisites

- Python 3.11 or later.
- [Ollama](https://ollama.com/) running locally with the `granite4.1:3b` model. No API keys are needed.

## Step 1. Set up your environment

You can run this notebook in [Colab](https://colab.research.google.com/), or download it and run locally. We recommend a [virtual environment](https://docs.python.org/3/library/venv.html) to avoid dependency conflicts.

In [ ]:
# Uncomment to install dependencies in Colab or a fresh environment
# !pip install mellea nest_asyncio pandas

### Set up a backend

This recipe uses a local Ollama server with the IBM Granite model, which needs no credentials. Start Ollama and pull the model before running the next cell:

```bash
ollama serve
ollama pull granite4.1:3b
```

To use watsonx.ai or another backend instead, change the `start_session` call in Step 2.

In [ ]:
import asyncio
import json
import re
import sqlite3
from typing import Literal

import nest_asyncio
import pandas as pd
import pydantic

from mellea import generative, req, simple_validate, start_session
from mellea.stdlib.components.react import MelleaTool
from mellea.stdlib.frameworks.react import react
from mellea.stdlib.context import ChatContext
from mellea.stdlib.components import Message

nest_asyncio.apply()  # Jupyter already runs an event loop; react() is async
m = start_session(backend_name="ollama", model_id="granite4.1:3b")
print("✓ Session started")

## Step 2. Create the support database

Three customers and four orders. Some tickets reference orders that don't exist, so the pipeline has to handle missing data.

In [ ]:
conn = sqlite3.connect(":memory:", check_same_thread=False)
conn.executescript("""
CREATE TABLE customers (
    customer_id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    tier TEXT NOT NULL
);
CREATE TABLE orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT REFERENCES customers(customer_id),
    item TEXT NOT NULL,
    status TEXT NOT NULL,
    estimated_delivery TEXT
);
""")

conn.executemany("INSERT INTO customers VALUES (?, ?, ?)", [
    ("C-100", "Priya Nair", "gold"),
    ("C-200", "Tom Becker", "standard"),
    ("C-300", "Lena Ortiz", "standard"),
])
conn.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", [
    ("O-1001", "C-100", "Noise-cancelling headphones", "delivered", "2026-11-10"),
    ("O-1002", "C-100", "USB-C dock", "shipped", "2026-11-18"),
    ("O-2001", "C-200", "Smart thermostat", "delayed", "2026-11-25"),
    ("O-3001", "C-300", "Standing desk", "processing", "2026-12-02"),
])
conn.commit()

display(pd.read_sql_query("SELECT * FROM orders", conn))

## Step 3. Define the tools

Tools are plain Python functions with type hints and docstrings. `MelleaTool.from_callable` builds the JSON schema the model sees from them. Each tool returns a string, so the agent can read it back.

In [ ]:
def lookup_order(order_id: str) -> str:
    """Look up an order by its ID (for example O-1001). Returns the item, status, and estimated delivery date, or a not-found message."""
    row = conn.execute(
        "SELECT order_id, customer_id, item, status, estimated_delivery FROM orders WHERE order_id = ?",
        (order_id.strip().upper(),),
    ).fetchone()
    if row is None:
        return f"No order found with ID {order_id}"
    return json.dumps(dict(zip(["order_id", "customer_id", "item", "status", "estimated_delivery"], row)))


def lookup_customer(customer_id: str) -> str:
    """Look up a customer by their ID (for example C-100). Returns the name and support tier, or a not-found message."""
    row = conn.execute(
        "SELECT customer_id, name, tier FROM customers WHERE customer_id = ?",
        (customer_id.strip().upper(),),
    ).fetchone()
    if row is None:
        return f"No customer found with ID {customer_id}"
    return json.dumps(dict(zip(["customer_id", "name", "tier"], row)))


TOOLS = [MelleaTool.from_callable(lookup_order), MelleaTool.from_callable(lookup_customer)]
print("✓ Tools:", [t.as_json_tool["function"]["name"] for t in TOOLS])

## Step 4. Agent 1: Classify the ticket

The first agent returns a typed `TicketTriage`. The category and priority are literal choices, so a free-text answer like "urgent-ish" can't pass.

In [ ]:
class TicketTriage(pydantic.BaseModel):
    category: Literal["order_status", "damaged_item", "refund", "account", "other"]
    priority: Literal["low", "medium", "high"]
    order_id: str | None = None  # The order ID mentioned in the ticket, if any
    reason: str


@generative
def classify_ticket(ticket_text: str) -> TicketTriage:
    """Classify a customer support ticket. Choose one category and one priority.
    - category: order_status for delivery questions, damaged_item for broken items, refund for money-back requests, account for login or profile issues, other for anything else.
    - priority: high if the customer is angry or the item is damaged, medium for delivery delays, low for general questions.
    - order_id: find any order ID in the ticket. Order IDs look like O-1001 (the letter O, a hyphen, then four digits). Copy it exactly as written, or return null only if the ticket contains no order ID.
    - reason: one short sentence explaining the choice.
    """


TICKET = (
    "Hi, this is Priya (customer C-100). My USB-C dock order O-1002 still says shipped, "
    "and the estimated date has passed. Can you tell me where it is?"
)

triage = classify_ticket(m, ticket_text=TICKET)
print(triage)


## Step 5. Agent 2: Research the order with tools

The research agent uses `react`, which lets the model call tools in a loop until it has an answer. It gets the classification from Step 4, so its goal is set by the typed output of the previous agent, not by free text.

In [ ]:
def research_ticket(triage: TicketTriage, ticket_text: str) -> str:
    """Run a tool-using agent that looks up the order and customer mentioned in the ticket."""
    order_hint = f" The order ID is {triage.order_id}." if triage.order_id else ""
    goal = (
        f"A support ticket was classified as '{triage.category}' with '{triage.priority}' priority.{order_hint}\n"
        f"Ticket: {ticket_text}\n"
        "Use the tools to look up the order and the customer. Report the order status, item, and estimated "
        "delivery date, using only what the tools return. If a tool says no order was found, say so."
    )
    context = ChatContext().add(Message("user", goal))
    thought, _ = asyncio.run(
        react(goal=goal, context=context, backend=m.backend, tools=TOOLS, loop_budget=6)
    )
    return str(thought.value)


research_notes = research_ticket(triage, TICKET)
print("Research notes:")
print(research_notes)


## Step 6. Agent 3: Draft the reply with requirements

The reply agent gets the typed triage and the research notes. Requirements run on every draft:

- The reply must state the order status the tools returned.
- It must not promise a refund or a specific delivery date unless the research notes include one.

A failed check triggers repair with the reason attached.

In [ ]:
def reply_mentions_real_status(reply: str) -> bool:
    """The reply must mention the status returned for the order."""
    status_words = {"shipped", "delivered", "delayed", "processing"}
    found = {w for w in status_words if w in reply.lower()}
    return bool(found) and all(w in research_notes.lower() for w in found)


def reply_makes_no_refund_promise(reply: str) -> bool:
    """The reply must not promise a refund."""
    return not re.search(r"\b(will|we'll|we will)\s+(issue|send|process)\s+(a\s+)?refund", reply, re.IGNORECASE)


reply = m.instruct(
    """Write a short, polite reply to the customer, using only the facts in the research notes.
    Name the order, state its current status, and give the estimated delivery date if one is listed.
    Return ONLY the reply text.""",
    grounding_context={"triage": triage.model_dump_json(), "research_notes": research_notes},
    requirements=[
        req("The reply must state the order status from the research notes", validation_fn=simple_validate(reply_mentions_real_status)),
        req("The reply must not promise a refund", validation_fn=simple_validate(reply_makes_no_refund_promise)),
    ],
    return_sampling_results=True,
)
print("Success:", reply.success)
print("\nDraft reply:")
print(str(reply.result))

## Step 7. Check the pipeline end to end

Run the same pipeline on two more tickets. One names an order that doesn't exist, and the other names an unknown customer. The pipeline should report the missing data instead of inventing an answer.

In [ ]:
TICKETS = {
    "delayed order": TICKET,
    "missing order": "Hi, I'm Tom (C-200). Where is order O-9999? It never arrived.",
    "damaged item": "Order O-3001 arrived with a cracked desk leg. I'm really upset about this.",
}

rows = []
for label, text in TICKETS.items():
    t = classify_ticket(m, ticket_text=text)
    # Check the database directly so the table reflects the data, not the model's wording
    order_exists = bool(t.order_id) and lookup_order(t.order_id).startswith("{")
    rows.append({
        "ticket": label,
        "category": t.category,
        "priority": t.priority,
        "order_id": t.order_id,
        "order_in_database": order_exists,
    })

summary = pd.DataFrame(rows)
display(summary)

## Step 8. Clean up

In [ ]:
conn.close()
print("✓ Database connection closed")

## Summary

The pipeline is three agents with typed handoffs:

- **Classify** returns a `TicketTriage`, so category and priority are limited to known values.
- **Research** calls tools by name and returns the results it found. Missing orders are reported, not invented.
- **Reply** is checked by requirements. A reply that states the wrong status or promises a refund is repaired before it's returned.

### Next steps

- Add a supervisor agent that routes tickets to different research tools by category.
- Add more tools, such as a refund-policy lookup, and check the reply against it.
- Connect the database to a real ticketing system through a tool.

## Complete Script

The full pipeline as one script. Start Ollama and pull `granite4.1:3b` first.

```python
import asyncio
import json
import re
import sqlite3
from typing import Literal

import nest_asyncio
import pandas as pd
import pydantic

from mellea import generative, req, simple_validate, start_session
from mellea.stdlib.components.react import MelleaTool
from mellea.stdlib.frameworks.react import react
from mellea.stdlib.context import ChatContext
from mellea.stdlib.components import Message

nest_asyncio.apply()  # Jupyter already runs an event loop; react() is async
m = start_session(backend_name="ollama", model_id="granite4.1:3b")
print("✓ Session started")

conn = sqlite3.connect(":memory:", check_same_thread=False)
conn.executescript("""
CREATE TABLE customers (
    customer_id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    tier TEXT NOT NULL
);
CREATE TABLE orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT REFERENCES customers(customer_id),
    item TEXT NOT NULL,
    status TEXT NOT NULL,
    estimated_delivery TEXT
);
""")

conn.executemany("INSERT INTO customers VALUES (?, ?, ?)", [
    ("C-100", "Priya Nair", "gold"),
    ("C-200", "Tom Becker", "standard"),
    ("C-300", "Lena Ortiz", "standard"),
])
conn.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", [
    ("O-1001", "C-100", "Noise-cancelling headphones", "delivered", "2026-11-10"),
    ("O-1002", "C-100", "USB-C dock", "shipped", "2026-11-18"),
    ("O-2001", "C-200", "Smart thermostat", "delayed", "2026-11-25"),
    ("O-3001", "C-300", "Standing desk", "processing", "2026-12-02"),
])
conn.commit()

display(pd.read_sql_query("SELECT * FROM orders", conn))

def lookup_order(order_id: str) -> str:
    """Look up an order by its ID (for example O-1001). Returns the item, status, and estimated delivery date, or a not-found message."""
    row = conn.execute(
        "SELECT order_id, customer_id, item, status, estimated_delivery FROM orders WHERE order_id = ?",
        (order_id.strip().upper(),),
    ).fetchone()
    if row is None:
        return f"No order found with ID {order_id}"
    return json.dumps(dict(zip(["order_id", "customer_id", "item", "status", "estimated_delivery"], row)))


def lookup_customer(customer_id: str) -> str:
    """Look up a customer by their ID (for example C-100). Returns the name and support tier, or a not-found message."""
    row = conn.execute(
        "SELECT customer_id, name, tier FROM customers WHERE customer_id = ?",
        (customer_id.strip().upper(),),
    ).fetchone()
    if row is None:
        return f"No customer found with ID {customer_id}"
    return json.dumps(dict(zip(["customer_id", "name", "tier"], row)))


TOOLS = [MelleaTool.from_callable(lookup_order), MelleaTool.from_callable(lookup_customer)]
print("✓ Tools:", [t.as_json_tool["function"]["name"] for t in TOOLS])

class TicketTriage(pydantic.BaseModel):
    category: Literal["order_status", "damaged_item", "refund", "account", "other"]
    priority: Literal["low", "medium", "high"]
    order_id: str | None = None  # The order ID mentioned in the ticket, if any
    reason: str


@generative
def classify_ticket(ticket_text: str) -> TicketTriage:
    """Classify a customer support ticket. Choose one category and one priority.
    - category: order_status for delivery questions, damaged_item for broken items, refund for money-back requests, account for login or profile issues, other for anything else.
    - priority: high if the customer is angry or the item is damaged, medium for delivery delays, low for general questions.
    - order_id: find any order ID in the ticket. Order IDs look like O-1001 (the letter O, a hyphen, then four digits). Copy it exactly as written, or return null only if the ticket contains no order ID.
    - reason: one short sentence explaining the choice.
    """


TICKET = (
    "Hi, this is Priya (customer C-100). My USB-C dock order O-1002 still says shipped, "
    "and the estimated date has passed. Can you tell me where it is?"
)

triage = classify_ticket(m, ticket_text=TICKET)
print(triage)


def research_ticket(triage: TicketTriage, ticket_text: str) -> str:
    """Run a tool-using agent that looks up the order and customer mentioned in the ticket."""
    order_hint = f" The order ID is {triage.order_id}." if triage.order_id else ""
    goal = (
        f"A support ticket was classified as '{triage.category}' with '{triage.priority}' priority.{order_hint}\n"
        f"Ticket: {ticket_text}\n"
        "Use the tools to look up the order and the customer. Report the order status, item, and estimated "
        "delivery date, using only what the tools return. If a tool says no order was found, say so."
    )
    context = ChatContext().add(Message("user", goal))
    thought, _ = asyncio.run(
        react(goal=goal, context=context, backend=m.backend, tools=TOOLS, loop_budget=6)
    )
    return str(thought.value)


research_notes = research_ticket(triage, TICKET)
print("Research notes:")
print(research_notes)


def reply_mentions_real_status(reply: str) -> bool:
    """The reply must mention the status returned for the order."""
    status_words = {"shipped", "delivered", "delayed", "processing"}
    found = {w for w in status_words if w in reply.lower()}
    return bool(found) and all(w in research_notes.lower() for w in found)


def reply_makes_no_refund_promise(reply: str) -> bool:
    """The reply must not promise a refund."""
    return not re.search(r"\b(will|we'll|we will)\s+(issue|send|process)\s+(a\s+)?refund", reply, re.IGNORECASE)


reply = m.instruct(
    """Write a short, polite reply to the customer, using only the facts in the research notes.
    Name the order, state its current status, and give the estimated delivery date if one is listed.
    Return ONLY the reply text.""",
    grounding_context={"triage": triage.model_dump_json(), "research_notes": research_notes},
    requirements=[
        req("The reply must state the order status from the research notes", validation_fn=simple_validate(reply_mentions_real_status)),
        req("The reply must not promise a refund", validation_fn=simple_validate(reply_makes_no_refund_promise)),
    ],
    return_sampling_results=True,
)
print("Success:", reply.success)
print("\nDraft reply:")
print(str(reply.result))

TICKETS = {
    "delayed order": TICKET,
    "missing order": "Hi, I'm Tom (C-200). Where is order O-9999? It never arrived.",
    "damaged item": "Order O-3001 arrived with a cracked desk leg. I'm really upset about this.",
}

rows = []
for label, text in TICKETS.items():
    t = classify_ticket(m, ticket_text=text)
    # Check the database directly so the table reflects the data, not the model's wording
    order_exists = bool(t.order_id) and lookup_order(t.order_id).startswith("{")
    rows.append({
        "ticket": label,
        "category": t.category,
        "priority": t.priority,
        "order_id": t.order_id,
        "order_in_database": order_exists,
    })

summary = pd.DataFrame(rows)
display(summary)

conn.close()
print("✓ Database connection closed")
```

## Running the Notebook

Start Ollama and pull the model, then run all cells top to bottom. The Step 7 summary runs three tickets through the pipeline.

## Key Takeaways

- Typed handoffs (`TicketTriage`) limit each agent's output to known values, so a bad category fails validation instead of reaching the next step.
- Tools return data the agent must report, and the summary checks the database directly rather than trusting the model's wording.
- Requirements on the reply (stated status, no refund promises) are checked before the draft is returned.
- Explicit order-ID extraction in the classifier makes the research step reliable.
- A missing order is reported as missing, not invented.

## Next Steps

- [Instruct-Validate-Repair](../InstructValidateRepair/InstructValidateRepair.ipynb): the validation pattern used for the reply step.
- [SQL Query Modernization](../SQLModernization/SQLModernization.ipynb): applies the same pattern to a database query.